In [ ]:
import json
import requests
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LinearRegression
from sklearn import metrics
%matplotlib inline
key = 'CBTgkcUJ'

In [ ]:
def dev(a,b):
    return math.sqrt(abs(a-b))

In [ ]:
originConditions = pd.read_csv('C:/Users/rohin/OneDrive/Desktop/Book1.csv')

In [ ]:
originConditions

In [ ]:
originConditions.iloc[0][1]

In [ ]:
analysis = {'Deaths': [originConditions.iloc[0][1]-originConditions.iloc[2][1], originConditions.iloc[2][1]-originConditions.iloc[6][1], originConditions.iloc[3][1]-originConditions.iloc[0][1], originConditions.iloc[4][1]-originConditions.iloc[1][1], originConditions.iloc[5][1]-originConditions.iloc[4][1]],
            'Temp_dev': [dev(originConditions.iloc[0][2],originConditions.iloc[2][2]), dev(originConditions.iloc[2][2],originConditions.iloc[6][2]), dev(originConditions.iloc[3][2],originConditions.iloc[0][2]), dev(originConditions.iloc[4][2],originConditions.iloc[1][2]), dev(originConditions.iloc[5][2],originConditions.iloc[4][2])],
           'Prec_dev': [dev(originConditions.iloc[0][3],originConditions.iloc[2][3]), dev(originConditions.iloc[2][3],originConditions.iloc[6][3]), dev(originConditions.iloc[3][3],originConditions.iloc[0][3]), dev(originConditions.iloc[4][3],originConditions.iloc[1][3]), dev(originConditions.iloc[5][3],originConditions.iloc[4][3])],
           'Pres_dev': [dev(originConditions.iloc[0][4],originConditions.iloc[2][4]), dev(originConditions.iloc[2][4],originConditions.iloc[6][4]), dev(originConditions.iloc[3][4],originConditions.iloc[0][4]), dev(originConditions.iloc[4][4],originConditions.iloc[1][4]), dev(originConditions.iloc[5][4],originConditions.iloc[4][4])]}

In [ ]:
analysis

In [ ]:
dataset = pd.DataFrame.from_dict(analysis)

In [ ]:
dataset

In [ ]:
X = dataset[['Temp_dev', 'Prec_dev', 'Pres_dev']].values
X1 = dataset[['Temp_dev', 'Prec_dev', 'Pres_dev']]

In [ ]:
y = dataset['Deaths'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [ ]:
regressor = LinearRegression()  
regressor.fit(X, y)

In [ ]:
coeff_df = pd.DataFrame(regressor.coef_, X1.columns, columns=['Coefficient'])  
coeff_df

In [ ]:
y_pred = regressor.predict(X_test)

In [ ]:
df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})

In [ ]:
print('Root Mean Squared Error:', np.sqrt(metrics.mean_squared_error(y_test, y_pred)))

In [ ]:
df.plot(kind='bar',figsize=(5,4))
plt.grid(which='major', linestyle='-', linewidth='0.5', color='green')
plt.grid(which='minor', linestyle=':', linewidth='0.5', color='black')
plt.show()

In [ ]:
def dataVer(lat,long):
    URLStation = f'https://api.meteostat.net/v1/stations/nearby?lat={lat}&lon={long}&limit=5&key={key}'
    rStation = requests.get(url = URLStation)
    nearbyStation = rStation.json()
    if len(nearbyStation['data']) == 0:
        print('nearby stations not available, please try different co-ordinates (try big cities)')
        return (1,0,0,0)
    else:
        for i in range(5):
            station=nearbyStation['data'][i]['id']
            URLStationData = f'https://api.meteostat.net/v1/history/monthly?station={station}&start=2019-01&end=2019-12&key={key}'
            rStationData = requests.get(url = URLStationData)
            stationData = rStationData.json()
            if len(stationData['data']) != 0:
                return (0,stationData['data'][9]['temperature_mean'],stationData['data'][9]['pressure'],stationData['data'][9]['precipitation'])
        print('monthly data from nearby stations is unavailable unfortunately, please try different co-ordinates')
        return (1,0,0,0)

In [ ]:
flag=1
while flag:
    print('+ve for North and East, -ve for South and West')
    latA = input('Latitude of A: ')
    longA = input('Longitude of A: ')
    if dataVer(latA,longA)[0] == 1:
        flag=1
    else:
        aMeanTemp = dataVer(latA,longA)[1]
        aPres = dataVer(latA,longA)[2]
        aPrec = dataVer(latA,longA)[3]
        flag=0

In [ ]:
flag=1
while flag:
    print('+ve for North and East, -ve for South and West')
    latB = input('Latitude of B: ')
    longB = input('Longitude of B: ')
    if dataVer(latB,longB)[0] == 1:
        flag=1
    else:
        bMeanTemp = dataVer(latB,longB)[1]
        bPres = dataVer(latB,longB)[2]
        bPrec = dataVer(latB,longB)[3]
        flag=0

In [ ]:
data_arr = np.array([[dev(aMeanTemp,bMeanTemp), dev(aPres,bPres), dev(aPrec,bPrec)]])

In [ ]:
data_arr

In [ ]:
pred = regressor.predict(data_arr)

In [ ]:
print(f'estimated deaths between A and B: {pred[0]} (-ve value implies increase in population)')